# MedCLIPSeg on Google Colab — setup, reproduction & test plan

Runs the official [MedCLIPSeg](https://github.com/HealthX-Lab/MedCLIPSeg) code (CVPR 2026) on a Colab GPU.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 is enough for everything in Tiers 1–2; use L4/A100 for training on the big datasets).

| Tier | What | Why | Rough time on T4 |
|---|---|---|---|
| 0 | Setup + sanity checks | Make sure env, data and weights are right | ~10 min |
| 1 | Evaluate the **released checkpoints** (BUSI → ID + 4 OOD breast-US sets) | Confirms the pipeline reproduces the paper *without training* | ~10 min |
| 2 | Cheap probes on the released checkpoint: prompt designs, shuffled prompts, MC samples, empty-mask audit | Baseline numbers + where the method is fragile → improvement ideas | ~20 min |
| 3 | **Train from scratch** (BUSI 10% → BUSI 100%) and compare with the released checkpoint | Confirms you can reproduce training — required before testing any modification | ~10 min + ~1 h |

Everything heavy (data, checkpoints, outputs) lives on the Colab VM's local disk; only small results (CSVs, logs) are copied to Google Drive at the end.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU"

In [ ]:
# ---- EDIT THESE ----
REPO_URL  = "https://github.com/HealthX-Lab/MedCLIPSeg.git"   # replace with YOUR fork, e.g. https://github.com/<you>/MedCLIPSeg.git
BRANCH    = "main"
USE_DRIVE = True                                             # keep zips + results on Google Drive between sessions
DRIVE_DIR = "/content/drive/MyDrive/MedCLIPSeg_colab"
SEED      = 666                                              # seed used for all released checkpoints
# --------------------
import os
REPO = "/content/MedCLIPSeg"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/zips", exist_ok=True); os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log --oneline -1

### Install dependencies
`requirements.txt` pins `torch==2.9.0` and `numpy<2.0`. On Colab we **keep the preinstalled torch/torchvision/numpy** (avoids a ~2 GB download and a forced runtime restart). The only NumPy‑2 incompatibility in the repo (`np.Inf` in `utils/SurfaceDice.py`) is patched in the next cell.
This combination was smoke‑tested end‑to‑end (train → test → eval) on Python 3.12 + torch 2.14 + NumPy 2.5.

In [ ]:
!grep -v -E "^(torch|torchvision|numpy)([=<>]|$)" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
import importlib
for m in ["torch", "torchvision", "numpy", "transformers", "monai", "timm", "cv2", "huggingface_hub", "pandas"]:
    print(f"{m:16s}", importlib.import_module(m).__version__)

### Compatibility patches (idempotent)
1. `np.Inf → np.inf` (NumPy 2) — otherwise `utils/eval.py` crashes.
2. DataLoader `num_workers=8` → number of Colab CPUs.
3. **Bug fix:** `train.py --resume` looked for the checkpoint in `trained_models/` instead of `trained_models/seed{N}/`, so resuming after a Colab disconnect silently restarted from epoch 0.

`git diff` shows exactly what changed. These are worth committing to your fork.

In [ ]:
%%writefile /content/colab_patches.py
"""Small compatibility/robustness patches for running MedCLIPSeg on Colab. Idempotent."""
import os, re, pathlib
def _read(path):
    raw = open(path, newline="").read(); crlf = "\r\n" in raw
    return raw.replace("\r\n", "\n"), crlf
def _write(path, s, crlf):
    open(path, "w", newline="").write(s.replace("\n", "\r\n") if crlf else s)
def patch(path, old, new):
    s, crlf = _read(path)
    if new in s: print(f"[skip] {path}: already patched"); return
    assert old in s, f"pattern not found in {path}: {old[:60]!r}"
    _write(path, s.replace(old, new), crlf); print(f"[ok]   {path}")
# 1) NumPy 2.x removed np.Inf (Colab ships NumPy 2) -> eval.py crashes without this
s, crlf = _read("utils/SurfaceDice.py")
_write("utils/SurfaceDice.py", re.sub(r"np\.Inf\b", "np.inf", s), crlf); print("[ok]   utils/SurfaceDice.py (np.Inf -> np.inf)")
# 2) DataLoader workers: repo hard-codes 8; Colab has 2-12 vCPUs
n = min(8, os.cpu_count() or 2)
patch("train.py", "num_workers=8,\n                            pin_memory=True,)", f"num_workers={n},\n                            pin_memory=True,)")
patch("train.py", "num_workers=8,\n                        pin_memory=True)", f"num_workers={n},\n                        pin_memory=True)")
# 3) --resume looked in the wrong folder (missing seed{N}/), so it silently restarted from epoch 0
patch("train.py",
      '            "trained_models",\n            f"{results_name}_latest.pth")',
      '            "trained_models",\n            f"seed{cfg.seed}",\n            f"{results_name}_latest.pth")')
patch("train.py", "checkpoint = torch.load(resume_path)", "checkpoint = torch.load(resume_path, map_location=cfg.MODEL.DEVICE, weights_only=False)")

In [ ]:
!python /content/colab_patches.py
!git diff --stat

In [ ]:
# Cache the UniMedCLIP backbone weights on Drive so they are downloaded only once
if USE_DRIVE:
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    if not os.path.islink("checkpoints"):
        !rm -rf checkpoints
        os.symlink(f"{DRIVE_DIR}/checkpoints", "checkpoints")
!ls -la checkpoints/ 2>/dev/null | head

### Data
Datasets come from the authors' Hugging Face repo (`TahaKoleilat/MedCLIPSeg`, one zip per dataset). The repo already contains the text prompts (`data/<DS>/Prompts_Folder`); the zips add `Train_Folder / Val_Folder / Test_Folder` with `img/` and `label/`.
If the download returns 401/403, open the dataset page on Hugging Face, accept the terms, and run `from huggingface_hub import login; login()` with a read token.

Groups used by the paper: **BUS** = BUSI → BUSBRA, BUSUC, BUID, UDIAT · **ENDO** = Kvasir → ColonDB, ClinicDB, CVC300, BKAI · **DERM** = ISIC → UWaterlooSkinCancer · **BRAIN** = BTMRI → BRISC. Data‑efficiency datasets: BUSI, BTMRI, ISIC, Kvasir, Covid19, EUS (EUS is on Google Drive and needs `utils/preprocess_EUS.py` — skip it for now).

In [ ]:
import shutil, subprocess, zipfile, glob
from huggingface_hub import hf_hub_download

ZIP_DIR = f"{DRIVE_DIR}/zips" if USE_DRIVE else "/content/zips"

def get_dataset(name):
    dst = f"data/{name}"
    if glob.glob(f"{dst}/Test_Folder/img/*"):
        print(f"[have] {name}"); return
    zpath = hf_hub_download(repo_id="TahaKoleilat/MedCLIPSeg", repo_type="dataset",
                            filename=f"{name}.zip", local_dir=ZIP_DIR)
    tmp = f"/content/_unzip/{name}"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zpath) as z: z.extractall(tmp)
    # zip layout may or may not have a top-level folder: find the dir that holds Test_Folder
    root = next(d for d, sub, _ in os.walk(tmp) if "Test_Folder" in sub or "Train_Folder" in sub)
    for item in os.listdir(root):
        src, out = os.path.join(root, item), os.path.join(dst, item)
        if item == "Prompts_Folder":            # keep the repo's prompt files, only add missing ones
            os.makedirs(out, exist_ok=True)
            for f in os.listdir(src):
                if not os.path.exists(os.path.join(out, f)): shutil.copy(os.path.join(src, f), out)
        elif os.path.isdir(src): shutil.copytree(src, out, dirs_exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"[ok]   {name}")

for ds in ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT"]:     # Tier 1-3 only need the breast-US group
    get_dataset(ds)

In [ ]:
# Sanity check: every image listed in the prompt files exists, and count empty ground-truth masks
import pandas as pd, numpy as np, cv2
def check(name):
    rows = []
    for split, xl in [("Train", "Train_text.xlsx"), ("Val", "Val_text.xlsx"), ("Test", "Test_text_original.xlsx")]:
        p = f"data/{name}/Prompts_Folder/{xl}"
        if not os.path.exists(p): continue
        df = pd.read_excel(p)
        folder = f"data/{name}/{split}_Folder"
        missing = sum(not os.path.exists(f"{folder}/img/{i}") for i in df["Image"])
        empty = sum(cv2.imread(f"{folder}/label/{g}", 0).max() == 0 for g in df["Ground Truth"]
                    if os.path.exists(f"{folder}/label/{g}"))
        rows.append(dict(dataset=name, split=split, prompts=len(df), missing_imgs=missing, empty_masks=empty))
    return rows
display(pd.DataFrame(sum([check(d) for d in ["BUSI", "BUSBRA", "BUSUC", "BUID", "UDIAT"]], [])))

### Helpers: evaluate + compare with the paper
`evaluate()` runs `test.py` (30 MC samples by default → binary mask + uncertainty map per image) then `utils/eval.py` (DSC + NSD@2px per image → CSV), and reads the CSV back.
It also reports DSC on **non‑empty ground truth only**: `eval.py` gives an empty prediction on an empty mask a perfect score of 1.0, which can inflate averages on datasets with many "normal" images.

In [ ]:
RESULTS = []
PAPER = {  # MedCLIPSeg DSC (%) from the paper: Tables S14–S17 (data efficiency) and Table 2 (domain generalisation)
    ("BUSI",10):68.66, ("BUSI",25):77.73, ("BUSI",50):81.48, ("BUSI",100):85.72,
    ("BTMRI",10):79.07, ("BTMRI",25):83.93, ("BTMRI",50):85.93, ("BTMRI",100):88.03,
    ("ISIC",10):90.35, ("ISIC",25):91.00, ("ISIC",50):91.97, ("ISIC",100):92.54,
    ("Kvasir",10):77.21, ("Kvasir",25):84.21, ("Kvasir",50):88.18, ("Kvasir",100):90.15,
    ("Covid19",10):79.73, ("Covid19",25):81.83, ("Covid19",50):82.97, ("Covid19",100):83.41,
    ("BUSBRA",100):75.06, ("BUSUC",100):84.37, ("BUID",100):78.99, ("UDIAT",100):74.64,
    ("ColonDB",100):71.90, ("ClinicDB",100):80.80, ("CVC300",100):80.82, ("BKAI",100):79.15,
    ("BRISC",100):80.92, ("UWaterlooSkinCancer",100):83.53,
}

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-6000:]); raise RuntimeError("command failed")
    return p.stdout

def evaluate(target, source=None, pct=100, prompt="original", out="outputs_medclipseg", opts="", tag="", seed=SEED):
    source = source or target
    sh(f"python test.py --config-file configs/{target}.yaml --output-dir {out} --source_dataset {source} "
       f"--data_percentage {pct} --prompt_design {prompt} --seed {seed} {opts}")
    sh(f"python utils/eval.py --config-file configs/{target}.yaml --output-dir {out} "
       f"--data_percentage {pct} --prompt_design {prompt} --seed {seed} {opts}")
    name = target if pct == 100 else f"{target}_{pct}"
    df = pd.read_csv(f"{out}/{name}/seg_results/seed{seed}/test_Prompt-{prompt}.csv")
    lab = f"data/{target}/Test_Folder/label"
    df["gt_empty"] = [cv2.imread(f"{lab}/{n}", 0).max() == 0 for n in df["Name"]]
    r = dict(tag=tag, source=source, target=target, pct=pct, prompt=prompt, opts=opts, n=len(df),
             DSC=float(100*df.DSC.mean()), NSD=float(100*df.NSD.mean()),
             DSC_nonempty=float(100*df.loc[~df.gt_empty, "DSC"].mean()), n_empty=int(df.gt_empty.sum()),
             paper_DSC=PAPER.get((target, pct)) if prompt == "original" and source in (target, "BUSI", "Kvasir", "ISIC", "BTMRI") else None)
    RESULTS.append(r); print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()})
    return df

def show_results():
    d = pd.DataFrame(RESULTS)
    d["delta"] = d.DSC - pd.to_numeric(d.paper_DSC)
    display(d.round(2))

## 1. Tier 1 — reproduce the paper from the released checkpoints (no training)
`utils/download_ckpt.py` puts the authors' checkpoint at `outputs_medclipseg/<DS>/trained_models/seed666/…_latest.pth`, which is exactly where `test.py` looks.
**Pass criterion:** each DSC within ~±1 point of the paper (small differences come from MC sampling noise and library versions).

In [ ]:
sh(f"python utils/download_ckpt.py --dataset BUSI --seed {SEED}")
evaluate("BUSI", tag="T1-ID")                                     # paper: 85.72
for tgt in ["BUSBRA", "BUSUC", "BUID", "UDIAT"]:                  # paper: 75.06 / 84.37 / 78.99 / 74.64
    evaluate(tgt, source="BUSI", tag="T1-OOD")
show_results()

Data‑efficiency checkpoints (BUSI trained on 10 / 25 / 50 % of the labels):

In [ ]:
for pct in [10, 25, 50]:
    sh(f"python utils/download_ckpt.py --dataset BUSI --data_percentage {pct} --seed {SEED}")
    evaluate("BUSI", pct=pct, tag="T1-eff")                       # paper: 68.66 / 77.73 / 81.48
show_results()

### Look at the outputs
Each test image produces a binary mask in `seg_results/` and a colour‑mapped, **per‑image min–max normalised** uncertainty map (predictive entropy of the MC mean) in `unc_results/`. Because of that normalisation, colours are not comparable across images.

In [ ]:
import matplotlib.pyplot as plt
def show_cases(target, source=None, pct=100, prompt="original", out="outputs_medclipseg", n=4, seed=SEED):
    source = source or target
    name = target if pct == 100 else f"{target}_{pct}"
    sub = f"MedCLIPSeg_unimedclip_ViT-B-16_Prompt-{prompt}"
    df = pd.read_csv(f"{out}/{name}/seg_results/seed{seed}/test_Prompt-{prompt}.csv").sort_values("DSC")
    texts = pd.read_excel(f"data/{target}/Prompts_Folder/Test_text_{prompt}.xlsx").set_index("Ground Truth")["Description"]
    picks = list(df.head(n // 2).Name) + list(df.tail(n - n // 2).Name)      # worst + best cases
    imgs = pd.read_excel(f"data/{target}/Prompts_Folder/Test_text_{prompt}.xlsx").set_index("Ground Truth")["Image"]
    fig, ax = plt.subplots(len(picks), 4, figsize=(13, 3.2 * len(picks)))
    for r, m in enumerate(picks):
        img = cv2.cvtColor(cv2.imread(f"data/{target}/Test_Folder/img/{imgs[m]}"), cv2.COLOR_BGR2RGB)
        gt = cv2.imread(f"data/{target}/Test_Folder/label/{m}", 0)
        pr = cv2.imread(f"{out}/{name}/seg_results/seed{seed}/{sub}/{m}", 0)
        un = cv2.cvtColor(cv2.imread(f"{out}/{name}/unc_results/seed{seed}/{sub}/{m}"), cv2.COLOR_BGR2RGB)
        for c, (im, t) in enumerate([(img, texts[m][:60]), (gt, "ground truth"),
                                     (pr, f"prediction  DSC={df.set_index('Name').DSC[m]:.2f}"), (un, "uncertainty")]):
            ax[r, c].imshow(im, cmap="gray"); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

show_cases("BUSI")
show_cases("UDIAT", source="BUSI")

## 2. Tier 2 — cheap probes on the released checkpoint (inference only)
These give you baseline numbers for the report and point at weaknesses worth improving.

### 2a. Prompt robustness (paper Table 4 / Appendix E)
The test prompts are generated from the ground‑truth masks (they state size, shape, count and **location** of the lesion), so the text carries information about the answer. The five provided designs are: `original`, `missing_location`, `underdescriptive`, `overdescriptive`, `contradictory`.
We add a sixth, **`shuffled`**: every image gets *another* test image's original prompt. If DSC barely drops, the model ignores the text; if it collapses, the model leans on oracle location cues.

In [ ]:
def make_shuffled(target, seed=0):
    p = f"data/{target}/Prompts_Folder/"
    df = pd.read_excel(p + "Test_text_original.xlsx")
    df["Description"] = df["Description"].sample(frac=1, random_state=seed).values
    df.to_excel(p + "Test_text_shuffled.xlsx", index=False)

for tgt in ["BUSI", "UDIAT"]:
    make_shuffled(tgt)
    for design in ["original", "missing_location", "underdescriptive", "overdescriptive", "contradictory", "shuffled"]:
        evaluate(tgt, source="BUSI", prompt=design, tag="T2a-prompt")
show_results()

### 2b. Monte‑Carlo samples vs. accuracy and runtime (paper Table S4)
`TEST.NUM_SAMPLES` = number of stochastic forward passes. Each pass reruns the *whole* CLIP encoder (the adapters sit inside every layer), so runtime grows linearly.

In [ ]:
import time
for k in [1, 5, 10, 30]:
    t0 = time.time()
    evaluate("BUSI", opts=f"TEST.NUM_SAMPLES {k}", tag=f"T2b-mc{k}")
    RESULTS[-1]["seconds"] = round(time.time() - t0, 1)
show_results()

### 2c. Seed / sampling noise
Re‑run the identical evaluation with different `--seed` values is **not** possible with released checkpoints (the seed is part of the checkpoint path). Instead copy the checkpoint under another seed folder to measure pure MC‑sampling variance at test time:

In [ ]:
src = f"outputs_medclipseg/BUSI/trained_models/seed{SEED}"
for s in [1, 2]:
    os.makedirs(f"outputs_medclipseg/BUSI/trained_models/seed{s}", exist_ok=True)
    shutil.copy(f"{src}/MedCLIPSeg_unimedclip_ViT-B-16_latest.pth", f"outputs_medclipseg/BUSI/trained_models/seed{s}/")
    evaluate("BUSI", seed=s, tag="T2c-mcnoise")
show_results()

## 3. Tier 3 — train from scratch and compare to the released checkpoint
Paper settings: batch 24, Adam lr 3e‑4, cosine schedule, **100 epochs**, last‑epoch checkpoint (`TEST.USE_LATEST: True`). Trainable parameters ≈ 18.7 M (adapters + mask head + upscaler; both CLIP encoders frozen).

* Batch 24 should fit on a T4 (estimated ~12–13 GB from a CPU memory profile). If you hit CUDA OOM, append `TRAIN.BATCH_SIZE 16` — but note it changes the in‑batch contrastive loss, so report it.
* Each checkpoint is ~1 GB (it stores the frozen CLIP weights too), and two are rewritten every epoch → keep `out` on local disk, not Drive.
* If Colab disconnects, re‑run the same cell with `--resume` added (works after the patch above).

**Step 1 – quick check:** BUSI 10 % (62 training images, a few minutes).

In [ ]:
OUT = "outputs_train"
!python train.py --config-file configs/BUSI.yaml --output-dir {OUT} --data_percentage 10 --seed {SEED}
evaluate("BUSI", pct=10, out=OUT, tag="T3-train")                # released ckpt vs your run, paper 68.66
show_results()

**Step 2 – full run:** BUSI 100 % (~1 h on T4, much less on A100), then the domain‑generalisation targets with *your* checkpoint.

In [ ]:
!python train.py --config-file configs/BUSI.yaml --output-dir {OUT} --seed {SEED}
evaluate("BUSI", out=OUT, tag="T3-train")
for tgt in ["BUSBRA", "BUSUC", "BUID", "UDIAT"]:
    evaluate(tgt, source="BUSI", out=OUT, tag="T3-train")
show_results()

In [ ]:
# Training curve from the log
import re
log = open(f"{OUT}/BUSI/trained_models/seed{SEED}/log.txt").read()
ep = [(int(a), float(b), float(c)) for a, b, c in re.findall(r"EPOCH: (\d+) \| Training Loss: ([\d.]+) \| Validation Loss: ([\d.]+)", log)]
dice = [float(x) for x in re.findall(r"(?:Dice: |→ )([\d.]+)\n", log)]
e, tr, va = zip(*ep)
plt.figure(figsize=(10, 3)); plt.subplot(1, 2, 1); plt.plot(e, tr, label="train"); plt.plot(e, va, label="val"); plt.legend(); plt.title("loss")
plt.subplot(1, 2, 2); plt.plot(range(1, len(dice) + 1), dice); plt.title("val Dice (1 MC sample)"); plt.show()

## 4. Save results to Drive

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
pd.DataFrame(RESULTS).to_csv(f"{DRIVE_DIR}/results/results_{stamp}.csv", index=False)
# per-image CSVs + logs (small); checkpoints are NOT copied (≈1 GB each) — copy one manually if you need it
!cd /content/MedCLIPSeg && find outputs_* -name "*.csv" -o -name "log.txt" | tar -czf {DRIVE_DIR}/results/csv_logs_{stamp}.tgz -T -
!ls -la {DRIVE_DIR}/results

## Next: extending to the other groups
Same pattern for the other domains (larger downloads, longer inference):
```python
for ds in ["Kvasir", "ColonDB", "ClinicDB", "CVC300", "BKAI"]: get_dataset(ds)
sh(f"python utils/download_ckpt.py --dataset Kvasir --seed {SEED}")
evaluate("Kvasir", tag="ENDO-ID")
for t in ["ColonDB", "ClinicDB", "CVC300", "BKAI"]: evaluate(t, source="Kvasir", tag="ENDO-OOD")
```
ISIC → UWaterlooSkinCancer and BTMRI → BRISC work the same way. BTMRI's test set contains many normal scans with empty masks, so watch the `DSC_nonempty` column there.